# Notebook 1: Exploring the Data

## Welcome! What Is "Training Data"?

Every AI system starts with **data**. Before an AI can do anything useful -- write text, recognize images, translate languages -- it first needs to study thousands (or even millions) of examples.

Think of it like learning to play music. A music student doesn't sit down and compose a symphony on day one. First, they listen to hundreds of songs. They study melodies, rhythms, and patterns. Over time, they develop an intuition for what "sounds right." Only then can they start creating their own music.

**AI works the same way.** We feed it a large collection of examples -- this is called **training data** -- and the AI studies those examples to find patterns. Once it has learned enough patterns, it can generate something new that resembles what it studied.

In this series of notebooks, our AI is going to learn from **Italian song lyrics**. By the end, it will be able to generate brand-new lyrics that sound like they could be real Italian songs. But first, we need to understand what our data looks like.

---

**What you will learn in this notebook:**
- What our raw dataset contains (songs, artists, metadata)
- Why we need to filter the data (not all songs are in Italian!)
- What the cleaned, processed data looks like
- Why the quality of data matters for AI

Let's dive in!

## Loading the Raw Dataset

Our dataset is a file called `the_italian_music_dataset.json`. It is stored in a format called **JSONL** (JSON Lines), where each line of the file is a separate song entry written in JSON format. Think of it like a spreadsheet where each row is a song, and each column holds a different piece of information about that song.

Let's load it up and see how many songs we have!

In [ ]:
import sys
import json
import random

# This lets us import from the project root if needed in later notebooks
sys.path.insert(0, '..')

# Load the raw dataset (one JSON object per line)
songs = []
with open('../data/the_italian_music_dataset.json', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            songs.append(json.loads(line))

print(f"Total number of songs in the dataset: {len(songs):,}")
print(f"\nLet's peek at the very first entry:")
print(json.dumps(songs[0], indent=2, ensure_ascii=False))

## What's Inside a Song?

Each song entry in our dataset is like a detailed card with several pieces of information:

| Field | What It Contains |
|-------|-----------------|
| **id_song** | A unique identifier for the song (like a barcode) |
| **artist** | Information about the artist: their name, genre (rock, pop, hip hop...), home region in Italy, and a popularity score |
| **musical_features** | Technical audio measurements like tempo, energy, danceability (from Spotify's analysis) |
| **lyrics** | The actual words of the song -- this is what our AI will learn from! |
| **song** | The title of the song |

Not every song has all fields (some are missing musical features, for example), but they all have **lyrics** -- and that is the most important part for us.

Let's look at a nicely formatted example:

In [ ]:
# Let's pick a song and display it in a readable way
sample_song = songs[0]

print("=" * 60)
print(f"  SONG:   {sample_song['song'].upper()}")
print(f"  ARTIST: {sample_song['artist']['name'].title()}")
print(f"  GENRE:  {sample_song['artist']['genre'].title()}")
print(f"  REGION: {sample_song['artist']['region'].title()}")
print("=" * 60)
print()
print("LYRICS (first 300 characters):")
print("-" * 40)
print(sample_song['lyrics'][:300] + "...")
print("-" * 40)
print()

# Show what fields are available
print("All fields in this entry:")
for key in sample_song.keys():
    print(f"  - {key}")

## The Language Challenge

Here is something important: even though this is called "The Italian Music Dataset," it contains songs in **multiple languages**. Many Italian artists record songs in English, and the dataset includes all of them.

This matters because our AI is supposed to learn Italian lyrics. If we feed it English lyrics too, it will get confused -- imagine a student trying to learn Italian but half their textbook is in English!

We need a way to automatically detect which language each song is in. Fortunately, there is a tool called `langdetect` that can analyze a piece of text and tell us what language it is. Let's see it in action:

In [ ]:
from langdetect import detect

# Let's test language detection on a few examples
examples = [
    ("Italian", "amare cantare sotto la pioggia nel cuore della notte"),
    ("English", "love is all you need when the sun goes down"),
    ("Spanish", "bailando bajo la luna en la playa de mis suenos"),
]

print("Language Detection Demo:")
print("=" * 60)
for label, text in examples:
    detected = detect(text)
    print(f"\n  Text:     \"{text}\"")
    print(f"  Expected: {label}")
    print(f"  Detected: {detected}")
print()

# Now let's check a sample of actual songs from our dataset
print("=" * 60)
print("Language detection on actual songs from our dataset:")
print("=" * 60)

# Check the first few songs
for i in [0, 1, 3, 7, 10]:
    if i < len(songs):
        song = songs[i]
        try:
            lang = detect(song['lyrics'])
        except:
            lang = "unknown"
        preview = song['lyrics'][:80] + "..."
        print(f"\n  \"{song['song']}\" by {song['artist']['name']}")
        print(f"  Detected language: {lang}")
        print(f"  Lyrics preview: {preview}")

As you can see, the language detector can tell Italian lyrics from English ones. Now let's count how many songs are in each language. This takes a moment since we are checking all 14,679 songs:

In [ ]:
from collections import Counter

# Detect the language of every song in the dataset
language_counts = Counter()
for song in songs:
    try:
        lang = detect(song['lyrics'])
        language_counts[lang] += 1
    except:
        language_counts['unknown'] += 1

# Show the results, sorted by frequency
print("Languages found in the dataset:")
print("=" * 40)
total = sum(language_counts.values())
for lang, count in language_counts.most_common(10):
    percentage = (count / total) * 100
    bar = "#" * int(percentage / 2)
    print(f"  {lang:>5}: {count:>5} songs ({percentage:5.1f}%) {bar}")

print(f"\n  Total: {total:>5} songs")
print(f"\nThe dataset has songs in {len(language_counts)} different detected languages.")
print(f"Italian ('it') songs make up about {language_counts.get('it', 0) / total * 100:.0f}% of the dataset.")

## Loading the Processed Data

The language filtering has already been done for us! The file `italian_lyrics.txt` contains only the Italian-language songs, with one song per line. This is the **cleaned data** that our AI will actually learn from.

Let's load it and explore what we are working with:

In [ ]:
# Load the processed Italian-only lyrics
with open('../data/italian_lyrics.txt', 'r', encoding='utf-8') as f:
    lyrics = [line.strip() for line in f if line.strip()]

print(f"Number of Italian songs (processed): {len(lyrics):,}")
print()

# Basic statistics
word_counts = [len(lyric.split()) for lyric in lyrics]
avg_words = sum(word_counts) / len(word_counts)
shortest = min(word_counts)
longest = max(word_counts)

print("Basic Statistics:")
print("=" * 40)
print(f"  Average song length: {avg_words:.0f} words")
print(f"  Shortest song:       {shortest} words")
print(f"  Longest song:        {longest} words")
print()

# Show 3 random samples
print("=" * 60)
print("Here are 3 random songs from the processed dataset:")
print("=" * 60)

random.seed(42)  # So we get the same samples every time
samples = random.sample(lyrics, 3)
for i, sample in enumerate(samples, 1):
    words = sample.split()
    print(f"\n--- Song {i} ({len(words)} words) ---")
    # Show first 150 characters for readability
    preview = sample[:150]
    if len(sample) > 150:
        preview += "..."
    print(preview)

## Visualizing the Data

Numbers are useful, but a picture is worth a thousand words. Let's create a **histogram** -- a type of chart that shows how many songs fall into different length ranges. This will help us see at a glance whether most songs are short, long, or somewhere in between.

In [ ]:
import matplotlib.pyplot as plt

# Create a histogram of song lengths
plt.figure(figsize=(10, 5))
plt.hist(word_counts, bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
plt.xlabel('Number of Words in a Song', fontsize=13)
plt.ylabel('Number of Songs', fontsize=13)
plt.title('How Long Are Italian Song Lyrics?', fontsize=15, fontweight='bold')

# Add a vertical line for the average
plt.axvline(avg_words, color='#E53935', linestyle='--', linewidth=2,
            label=f'Average: {avg_words:.0f} words')
plt.legend(fontsize=12)

plt.tight_layout()
plt.show()

print(f"Most songs are between 50 and 300 words long.")
print(f"The average song has about {avg_words:.0f} words.")

## Why Data Quality Matters

There is a famous saying in computer science: **"Garbage in, garbage out."**

What does this mean? Simply put: **an AI is only as good as the data it learns from.** If you feed it messy, incorrect, or biased data, it will produce messy, incorrect, or biased results.

Here are some real-world examples of why data quality matters:

- **Mistakes in the data**: If many of our lyrics had typos or garbled text, the AI would learn to produce garbled text too. It has no way of knowing what is "correct" -- it just copies the patterns it sees.

- **Missing diversity**: If our dataset only contained love ballads, the AI would only know how to write love ballads. It would never produce a protest song or a party anthem, because it never saw one. The data you choose **shapes and limits** what the AI can do.

- **Bias**: This is where things get serious. If the training data over-represents certain viewpoints, styles, or demographics, the AI will reflect those same imbalances. This is how bias enters AI systems -- not through intentional programming, but through the data the AI learns from.

This is why the first step in any AI project is always to **understand your data** -- which is exactly what we are doing in this notebook!

## Key Takeaways

Let's recap what we have learned in this notebook:

- **Every AI starts with data.** Before an AI can generate anything, it needs to study a large collection of examples. Our examples are Italian song lyrics.

- **Raw data is messy.** Our original dataset has 14,679 songs, but not all of them are in Italian. We had to filter by language to get clean, relevant data.

- **Data must be cleaned and filtered.** Using language detection, we narrowed our dataset down to approximately **9,135 Italian-language songs**. This is our training data.

- **The data you choose shapes what the AI can do.** If we only gave the AI rock lyrics, it would only know rock. Our dataset includes multiple genres (rock, pop, hip hop, and more), giving the AI a richer understanding of Italian music.

- **Quality matters more than quantity.** It is better to have 9,135 clean, relevant examples than 14,679 noisy ones. This is true for all AI systems, from chatbots to self-driving cars.

## What's Next?

We have explored our data and we know what we are working with: thousands of Italian song lyrics, cleaned and ready for our AI to study.

But here is a question: **How does an AI actually "read" text?**

Computers do not understand words the way we do. They work with numbers. So before we can train an AI on lyrics, we need to convert those words into numbers. This process is called **tokenization**, and it is a fascinating (and surprisingly important) step.

**In the next notebook, we will discover tokenization** -- how an AI breaks text into small pieces and turns those pieces into numbers it can work with. You might be surprised by how it decides to split up words!

---

*Continue to [Notebook 2: How Does an AI Read Text? (Tokenization)](02_tokenization.ipynb)*